# No-Split Comparison (Illustrative Only — Not a Valid Result)

**This notebook exists to demonstrate a mistake, not to report a result.** Every model below is fit and evaluated on the *same* full dataset - no train/test split, no held-out data at all. That means every R² here is inflated by a real form of leakage: each model's parameters are solved to directly minimize error against the exact target values it is then "tested" against. This is not a genuine measure of forecasting skill - see the honest, properly-split results in `capstone-btc-volatility.ipynb` for the real numbers.

**Purpose:** to show, side by side, exactly how much each model's apparent performance is inflated once you remove the one safeguard (a genuine, chronological train/test split) that makes an R² trustworthy. Same 3 models and baseline as the main notebook - GARCH, HAR-RV-X (Final), EWMA, and Naive Persistence - same feature engineering, same hyperparameter choices (not re-tuned here) - the *only* thing that changes is that "test" data is the same data used for fitting.

In [1]:
import numpy as np
import pandas as pd
import warnings
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from arch import arch_model

RANDOM_STATE = 42
ROLLING_WINDOW = 24 * 7   # 168 hours = 7 days
HORIZON = ROLLING_WINDOW
HOURS_PER_DAY = 24
CSV_PATH = '../data/btc_hourly_ohlcv.csv'

def evaluate(y_true, y_pred, label):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, np.nan, y_true))) * 100
    print(f'{label:40s} RMSE={rmse:.6f}  MAE={mae:.6f}  R2={r2:.4f}  MAPE={mape:.2f}%')
    return {'Model': label, 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape}

results = []

## 1. Load Data & Build Features
Identical to the main notebook's Data Cleaning -> Technical Indicators -> Preprocessing steps.

In [2]:
df_raw = pd.read_csv(CSV_PATH, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
df = df_raw.copy()
df['hourly_return'] = df['Close'].pct_change()
df['log_return'] = np.log(df['Close'] / df['Close'].shift(1))
df['volume_chg'] = df['Volume'].pct_change()
df[['hourly_return', 'log_return', 'volume_chg']] = df[['hourly_return', 'log_return', 'volume_chg']].replace([np.inf, -np.inf], np.nan)
df['volatility_7d'] = df['hourly_return'].rolling(ROLLING_WINDOW).std()
df['ma_7d'] = df['Close'].rolling(ROLLING_WINDOW).mean()

def compute_indicators(df):
    close, high, low = df['Close'], df['High'], df['Low']
    ema_12 = close.ewm(span=12, adjust=False).mean()
    ema_26 = close.ewm(span=26, adjust=False).mean()
    df['ema_12'] = ema_12; df['ema_26'] = ema_26
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    df['atr_14'] = tr.rolling(14).mean()
    df['atr_pct'] = df['atr_14'] / close
    bb_mid = close.rolling(20).mean(); bb_std = close.rolling(20).std()
    df['bb_width'] = ((bb_mid + 2 * bb_std) - (bb_mid - 2 * bb_std)) / bb_mid
    macd_line = ema_12 - ema_26
    df['macd_hist'] = macd_line - macd_line.ewm(span=9, adjust=False).mean()
    return df

df = compute_indicators(df)
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'hourly_return', 'log_return', 'volatility_7d',
                 'ma_7d', 'volume_chg', 'ema_12', 'ema_26', 'rsi_14', 'atr_14', 'atr_pct', 'bb_width', 'macd_hist']
df_clean = df.dropna(subset=feature_cols).reset_index(drop=True)
df_clean_dated = df_clean.set_index('Date')

df_model = df_clean.copy()
df_model['target'] = df_model['volatility_7d'].shift(-HORIZON)
df_model = df_model.dropna(subset=['target']).reset_index(drop=True)

print(f'df_model: {len(df_model):,} rows, {df_model["Date"].min().date()} to {df_model["Date"].max().date()}')
print('NOTE: no train/test split is performed anywhere below.')

df_model: 72,944 rows, 2017-08-24 to 2025-12-24
NOTE: no train/test split is performed anywhere below.


## 2. Naive Persistence (Baseline)
No fitting involved - just holds the most recently known value constant. Included here mainly for scale/reference; a parameter-free baseline can't "leak" the way a fitted model can, but it's still evaluated over the exact same full dataset as everything else below, for a fair comparison.

In [3]:
naive_pred_full = df_model['volatility_7d']  # "next value = current value", no fitting
results.append(evaluate(df_model['target'], naive_pred_full, 'Naive Persistence (no split)'))

Naive Persistence (no split)             RMSE=0.003350  MAE=0.002108  R2=0.3130  MAPE=30.45%


## 3. GARCH — Fit on the Full Return Series, Forecast From Every Origin
The real leakage mechanism here: GARCH's ω/α/β/γ parameters are estimated by maximum likelihood over the **entire** return series - including hours far in the future relative to many of the forecast origins below. A forecast made "as of 2018" ends up using parameters partly shaped by 2024-2025 data it could never have had access to in reality.

In [4]:
returns_pct_full = df_clean_dated['hourly_return'].dropna() * 100

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    garch_full = arch_model(returns_pct_full, vol='GARCH', p=1, o=1, q=1, dist='t')
    garch_full_fit = garch_full.fit(disp='off')
    print(f'GARCH (GJR, t) fit on ALL {len(returns_pct_full):,} returns. AIC={garch_full_fit.aic:.2f}')

    # Forecast from every possible origin in the dataset (not just the tail 20%) - this is the
    # no-split analogue of the main notebook's rolling-origin HORIZON-step forecast.
    garch_fixed_full = garch_full.fix(garch_full_fit.params)
    garch_forecast_full = garch_fixed_full.forecast(horizon=HORIZON, start=0, reindex=False)

garch_pred_variance = garch_forecast_full.variance.values[:, -1]
garch_pred_sigma = np.sqrt(garch_pred_variance) / 100

# Align GARCH's forecast index (positional, starting from returns_pct_full's own index) back
# onto df_model's target - both are keyed by Date, so map through returns_pct_full's index.
garch_pred_series = pd.Series(garch_pred_sigma, index=returns_pct_full.index[-len(garch_pred_sigma):])
garch_aligned = df_model['Date'].map(garch_pred_series)
valid_garch = garch_aligned.notna()
results.append(evaluate(df_model.loc[valid_garch, 'target'], garch_aligned[valid_garch], 'GARCH (GJR,t) - no split'))

GARCH (GJR, t) fit on ALL 73,112 returns. AIC=108215.22


GARCH (GJR,t) - no split                 RMSE=0.004800  MAE=0.003981  R2=-0.4098  MAPE=80.06%


## 4. EWMA — Lambda Tuned and Evaluated on the Full Series
The leakage here: lambda is chosen to minimize error against the *entire* series at once - including using future volatility spikes to inform how reactive/smooth the estimator should be, rather than only ever seeing the past.

In [5]:
def ewma_sigma(returns, lam):
    r2 = returns.values ** 2
    n = len(r2)
    sigma2 = np.full(n, np.nan)
    valid = np.flatnonzero(~np.isnan(r2))
    if len(valid) == 0:
        return pd.Series(sigma2, index=returns.index)
    start = valid[0]
    sigma2[start] = r2[start]
    for t in range(start + 1, n):
        prev = sigma2[t - 1] if not np.isnan(sigma2[t - 1]) else 0
        sigma2[t] = lam * prev + (1 - lam) * (r2[t - 1] if not np.isnan(r2[t - 1]) else 0)
    return pd.Series(np.sqrt(sigma2), index=returns.index)

returns_frac_full = df_clean_dated['hourly_return']

# Tune lambda against the WHOLE series (no held-out fold) - this is the leak: lambda is chosen
# using knowledge of volatility spikes that, in a real deployment, wouldn't have happened yet.
lambda_grid = np.arange(0.85, 0.999, 0.01)
best_lambda_ns, best_rmse_ns = None, np.inf
for lam in lambda_grid:
    sigma = ewma_sigma(returns_frac_full, lam)
    aligned = df_model['Date'].map(sigma)
    valid = aligned.notna()
    rmse = np.sqrt(mean_squared_error(df_model.loc[valid, 'target'], aligned[valid]))
    if rmse < best_rmse_ns:
        best_lambda_ns, best_rmse_ns = lam, rmse

print(f'Best lambda (tuned on full series, no split): {best_lambda_ns:.3f}')
ewma_sigma_full = ewma_sigma(returns_frac_full, best_lambda_ns)
ewma_aligned = df_model['Date'].map(ewma_sigma_full)
valid_ewma = ewma_aligned.notna()
results.append(evaluate(df_model.loc[valid_ewma, 'target'], ewma_aligned[valid_ewma], f'EWMA (lambda={best_lambda_ns:.2f}) - no split'))

Best lambda (tuned on full series, no split): 0.990
EWMA (lambda=0.99) - no split            RMSE=0.003177  MAE=0.002029  R2=0.3825  MAPE=29.68%


## 5. HAR-RV-X (Final) — Ridge Fit and Evaluated on the Full Dataset
The clearest leakage of the three: Ridge directly solves for coefficients that minimize squared error against the exact target values used to "test" it - see the previous conversation's isolated comparison (0.2419 honest vs. 0.2883 no-split, log target) for the quantified gap this produces.

In [6]:
har_df = pd.DataFrame(index=df_model.index)
har_df['RV_day'] = df_model['volatility_7d'].shift(HOURS_PER_DAY)
har_df['RV_week'] = df_model['volatility_7d'].shift(HOURS_PER_DAY).rolling(HOURS_PER_DAY * 7).mean()
har_df['RV_month'] = df_model['volatility_7d'].shift(HOURS_PER_DAY).rolling(HOURS_PER_DAY * 30).mean()
har_df['ATR_X'] = df_model['atr_pct'].shift(1)
har_df['target'] = df_model['target']

ewma_tuned_sigma = ewma_sigma(returns_frac_full, 0.995)  # main notebook's own tuned (honest) lambda
har_df['EWMA_X'] = df_model['Date'].map(ewma_tuned_sigma).shift(1)
har_df['SKEW_X'] = df_model['hourly_return'].rolling(ROLLING_WINDOW).skew().shift(HOURS_PER_DAY)

har_final_df = har_df.dropna()
har_final_cols = ['RV_day', 'RV_week', 'RV_month', 'ATR_X', 'EWMA_X', 'SKEW_X']
ridge_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}

# Tuned AND evaluated on the same full dataset - no split at any stage
gs_nosplit = GridSearchCV(Ridge(random_state=RANDOM_STATE), ridge_grid, cv=TimeSeriesSplit(n_splits=5),
                           scoring='neg_root_mean_squared_error', n_jobs=-1)
gs_nosplit.fit(har_final_df[har_final_cols], np.log(har_final_df['target']))
har_pred_nosplit = np.exp(gs_nosplit.best_estimator_.predict(har_final_df[har_final_cols]))
print(f'Best alpha: {gs_nosplit.best_params_}')
results.append(evaluate(har_final_df['target'], har_pred_nosplit, 'HAR-RV-X (Final, log target) - no split'))

Best alpha: {'alpha': 0.001}
HAR-RV-X (Final, log target) - no split  RMSE=0.003301  MAE=0.001794  R2=0.2883  MAPE=26.29%


## 5b. HAR-RV-X (Final) — Same, But *Also* Without the Log Transform
**This is the least legitimate number in the entire project.** Section 5 removed one safeguard (the train/test split). This removes a second, independent one on top of it: no log transform on the target either, evaluated with a raw-scale squared-error loss that lets Ridge chase the few extreme, high-volatility outlier hours in-sample without any penalty for failing to generalize. Kept here, clearly labeled, purely to show how far a number can be pushed by stacking multiple unexamined shortcuts - not because it means anything about the model's real forecasting ability. The honest number is still 0.2419 (Section 6); this one exists to be recognized and rejected, not cited.

In [7]:
# Same features, same no-split evaluation as Section 5 - the ONLY change is fitting on the
# RAW target instead of log(target). Do not report this number; see the markdown above.
gs_nosplit_raw = GridSearchCV(Ridge(random_state=RANDOM_STATE), ridge_grid, cv=TimeSeriesSplit(n_splits=5),
                               scoring='neg_root_mean_squared_error', n_jobs=-1)
gs_nosplit_raw.fit(har_final_df[har_final_cols], har_final_df['target'])
har_pred_nosplit_raw = gs_nosplit_raw.best_estimator_.predict(har_final_df[har_final_cols])
print(f'Best alpha: {gs_nosplit_raw.best_params_}')
results.append(evaluate(har_final_df['target'], har_pred_nosplit_raw,
                         'HAR-RV-X (Final, RAW target) - no split [NOT VALID]'))

Best alpha: {'alpha': 0.01}
HAR-RV-X (Final, RAW target) - no split [NOT VALID] RMSE=0.002692  MAE=0.001720  R2=0.5268  MAPE=27.57%


## 6. Comparison — No-Split Numbers vs. the Honest, Properly-Split Numbers

In [8]:
results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
print(results_df.to_string(index=False))

print()
print('Honest (properly split) numbers from capstone-btc-volatility.ipynb, for comparison:')
honest_reference = pd.DataFrame([
    {'Model': 'Naive Persistence (honest)', 'RMSE': 0.001649, 'R2': -0.0899},
    {'Model': 'GARCH (GJR,t) - Tuned (honest)', 'RMSE': 0.003150, 'R2': -2.9773},
    {'Model': 'EWMA (lambda=0.995) - Tuned (honest)', 'RMSE': 0.001532, 'R2': 0.0593},
    {'Model': 'HAR-RV-X (Final) (honest)', 'RMSE': 0.001367, 'R2': 0.2419},
])

print(f"\nHAR-RV-X (Final, RAW target, no split) reaches R2={results_df.loc[results_df['Model'].str.contains('RAW'), 'R2'].values[0]:.4f} "
      f"- but only by stacking two unexamined shortcuts at once (no split + no transform).\n"
      f"It is not comparable to any of the honest numbers above and should never be reported as-is.")
print(honest_reference.to_string(index=False))

                                              Model     RMSE      MAE        R2      MAPE
HAR-RV-X (Final, RAW target) - no split [NOT VALID] 0.002692 0.001720  0.526802 27.570970
                      EWMA (lambda=0.99) - no split 0.003177 0.002029  0.382464 29.677480
            HAR-RV-X (Final, log target) - no split 0.003301 0.001794  0.288290 26.291081
                       Naive Persistence (no split) 0.003350 0.002108  0.313036 30.451753
                           GARCH (GJR,t) - no split 0.004800 0.003981 -0.409802 80.059339

Honest (properly split) numbers from capstone-btc-volatility.ipynb, for comparison:

HAR-RV-X (Final, RAW target, no split) reaches R2=0.5268 - but only by stacking two unexamined shortcuts at once (no split + no transform).
It is not comparable to any of the honest numbers above and should never be reported as-is.
                               Model     RMSE      R2
          Naive Persistence (honest) 0.001649 -0.0899
      GARCH (GJR,t) - Tuned (hones

## Conclusion — Why None of the Numbers Above Should Ever Be Reported
Every model in Sections 2-5 was fit and/or tuned using the exact same data it was then scored against - a direct, quantifiable form of leakage, not a stylistic choice. Compare each no-split R² to its honest counterpart in Section 6: every single model looks meaningfully better here than it does under a genuine chronological train/test split, precisely because "how well did I do" and "what did I train on" are the same question in this notebook, and different questions in the real one.

This notebook is kept in the project specifically as a demonstration - a clear, quantified answer to "what would happen if we hadn't split the data," so that the honest numbers in `capstone-btc-volatility.ipynb` can be shown, side by side, as the deliberate, correct choice rather than an unexamined default.

**One number here deserves its own warning: Section 5b's HAR-RV-X (Final, RAW target) reaches R²≈0.53** by stacking no-split *and* no-log-transform together - both shortcuts at once. That is the highest number anywhere in this notebook, and the least trustworthy: it isn't measuring forecasting skill at all, just how far an in-sample, unregularized-by-honesty fit can be pushed. If this number is ever shown to anyone, it must be shown with this exact context attached, never on its own.